# NCBI Assembly Manifest Generation (Phase 1)

Downloads the current NCBI assembly summary from FTP, compares it against a
previous snapshot, and produces:

- `transfer_manifest.txt` — assemblies to download in Phase 2
- `removed_manifest.txt` — assemblies to archive in Phase 3
- `diff_summary.json` — human-readable summary of changes

All filtering (prefix range, limit) is applied here so downstream phases
receive a final, pre-filtered manifest.

Optionally verifies candidates against the S3 Lakehouse (`STORE_BUCKET`) so
assemblies that were already downloaded and promoted are pruned from the
transfer manifest.

## Path formats quick reference

| Suffix in variable name | Format | Example |
|-------------------------|--------|---------|
| `_URI` | `s3://bucket/key/…` | `s3://cdm-lake/staging/run1/` |
| `_BUCKET` | bucket name only | `cdm-lake` |
| `_KEY_PREFIX` | S3 key prefix (no scheme/bucket) | `tenant-general-warehouse/kbase/datasets/ncbi/` |
| `_DIR` / `_PATH` | local filesystem path | `output/removed_manifest.txt` |

Lakehouse object: `s3://{STORE_BUCKET}/{STORE_KEY_PREFIX}raw_data/…/{filename}`

In [ ]:
"""Imports and S3 client initialisation."""

from __future__ import annotations

import json
from pathlib import Path

from cdm_data_loaders.ncbi_ftp.assembly import FTP_HOST
from cdm_data_loaders.ncbi_ftp.manifest import (
    AssemblyRecord,
    compute_diff,
    download_assembly_summary,
    filter_by_prefix_range,
    parse_assembly_summary,
    write_diff_summary,
    write_removed_manifest,
    write_transfer_manifest,
    write_updated_manifest,
)
from cdm_data_loaders.utils.s3 import get_s3_client, split_s3_path

In [ ]:
"""Configure parameters."""

# Which NCBI database to sync: "refseq" or "genbank"
DATABASE = "refseq"

# Accession prefix filtering (3-digit, inclusive). Set to None to skip.
PREFIX_FROM: str | None = "900"  # e.g. "000"
PREFIX_TO: str | None = "900"  # e.g. "003"

# Maximum number of new/updated assemblies to include (None = unlimited)
LIMIT: int | None = 10

# Previous assembly summary snapshot
# format: s3:// URI  (e.g. "s3://cdm-lake/.../assembly_summary_refseq_prev.txt")
PREVIOUS_SUMMARY_URI: str | None = None

# S3 location where the new snapshot will be uploaded after diffing
# format: s3:// URI
SNAPSHOT_UPLOAD_URI: str | None = None

# Verify candidates against the S3 Lakehouse — prune assemblies already present.
# Set STORE_BUCKET to your bucket name to enable, or None to skip.
# STORE_KEY_PREFIX should point to the directory containing `raw_data/`.
# format: bucket name (no s3:// scheme)
STORE_BUCKET: str | None = "cdm-lake"
# format: S3 key prefix within STORE_BUCKET (no scheme, no bucket)
STORE_KEY_PREFIX = "tenant-general-warehouse/kbase/datasets/ncbi/"

# Local output directory for manifest files
# format: local directory path
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Database: {DATABASE}")
print(f"Prefix range: {PREFIX_FROM} -> {PREFIX_TO}")
print(f"Limit: {LIMIT}")
print(f"Verify against S3: {STORE_BUCKET or 'disabled'}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
"""Download current assembly summary from NCBI FTP."""

raw_summary = download_assembly_summary(database=DATABASE, ftp_host=FTP_HOST)
current = parse_assembly_summary(raw_summary)
print(f"Parsed {len(current)} assemblies from current {DATABASE} summary")

In [ ]:
"""Optional: Bootstrap baseline by scanning current store (if no previous summary available).

If you have a pre-populated S3 store but no previous assembly summary snapshot,
you can scan the store to generate a synthetic summary. This becomes the baseline
for the diff.

Set SCAN_STORE=True below to enable. The scan will:
  1. List all objects under STORE_BUCKET/STORE_KEY_PREFIX
  2. Extract accessions and use earliest LastModified as seq_rel_date (conservative)
  3. Build AssemblyRecord for each assembly found
  4. Save to LOCAL_SYNTHETIC_SUMMARY for re-use in future runs

Typical use case: First run against 500K+ existing assemblies. Scanning takes
~5 minutes instead of ~6 days of checksum verification.
"""

SCAN_STORE = True  # Set to True to scan your store
LOCAL_SYNTHETIC_SUMMARY = Path("output/synthetic_summary_from_store.txt")

if SCAN_STORE and STORE_BUCKET:
    from cdm_data_loaders.ncbi_ftp.manifest import scan_store_to_synthetic_summary
    from IPython.display import display
    from tqdm.notebook import tqdm

    print(f"Scanning s3://{STORE_BUCKET}/{STORE_KEY_PREFIX} for existing assemblies ...")
    progress = tqdm(unit="assembly", desc="Scanning store", leave=True)
    display(progress)

    def _track_scan(count: int, acc: str) -> None:
        progress.n = count
        progress.set_postfix(acc=acc, refresh=False)
        progress.refresh()

    synthetic = scan_store_to_synthetic_summary(STORE_BUCKET, STORE_KEY_PREFIX, progress_callback=_track_scan)
    progress.refresh()

    print(f"Found {len(synthetic)} assemblies in store")

    # Save synthetic summary to file for future runs
    with LOCAL_SYNTHETIC_SUMMARY.open("w") as f:
        for acc in sorted(synthetic.keys()):
            rec = synthetic[acc]
            f.write(
                f"{rec.accession}\t.\t.\t.\t.\t.\t.\t.\t.\t.\t{rec.status}\t.\t.\t.\t{rec.seq_rel_date}\t.\t.\t.\t.\t{rec.ftp_path}\t.\n"
            )
    print(f"Saved synthetic summary to {LOCAL_SYNTHETIC_SUMMARY}")

    # Use it as the previous baseline
    previous = synthetic
else:
    if SCAN_STORE:
        print("SCAN_STORE=True but STORE_BUCKET not set. Skipping.")
    print("Skipping store scan (SCAN_STORE=False). Will load/use PREVIOUS_SUMMARY_URI instead.")


In [ ]:
"""Load previous summary from S3 (or from synthetic scan, or start fresh).

If you ran the store scan in the previous cell, SCAN_STORE=True above will have
set `previous` already. Otherwise, try to load from PREVIOUS_SUMMARY_URI.
"""

if "previous" not in locals() or previous is None:
    # Store scan didn't run, or was skipped. Try to load from S3.
    if PREVIOUS_SUMMARY_URI:
        s3 = get_s3_client()
        bucket, key = split_s3_path(PREVIOUS_SUMMARY_URI)
        resp = s3.get_object(Bucket=bucket, Key=key)
        prev_text = resp["Body"].read().decode("utf-8")
        previous = parse_assembly_summary(prev_text)
        print(f"Loaded {len(previous)} assemblies from previous snapshot")
    else:
        print("No previous snapshot and SCAN_STORE=False — all current 'latest' assemblies will be marked as new")
        previous = None


In [ ]:
"""Compute diff and apply prefix filter."""

# Filter current assemblies by prefix range
filtered = filter_by_prefix_range(current, prefix_from=PREFIX_FROM, prefix_to=PREFIX_TO)
print(f"After prefix filter: {len(filtered)} assemblies")

# Also filter previous if present
filtered_prev = filter_by_prefix_range(previous, prefix_from=PREFIX_FROM, prefix_to=PREFIX_TO) if previous else None

# Compute diff
diff = compute_diff(filtered, previous_assemblies=filtered_prev)

print(f"New:        {len(diff.new)}")
print(f"Updated:    {len(diff.updated)}")
print(f"Replaced:   {len(diff.replaced)}")
print(f"Suppressed: {len(diff.suppressed)}")
print(f"Total to transfer: {len(diff.new) + len(diff.updated)}")
print(f"Total to remove:   {len(diff.replaced) + len(diff.suppressed)}")

In [ ]:
"""Verify candidates against S3 Lakehouse, then apply LIMIT.

Verification (optional): for each candidate, fetch md5checksums.txt from
NCBI FTP and compare against md5 metadata on existing S3 objects.
Assemblies already present with matching checksums are pruned.

LIMIT is applied *after* verification so the cap counts only assemblies
that genuinely need downloading.
"""

# -- Verify against Lakehouse --
if STORE_BUCKET:
    from cdm_data_loaders.ncbi_ftp.manifest import verify_transfer_candidates
    from IPython.display import display
    from tqdm.notebook import tqdm

    candidates = diff.new + diff.updated
    total = len(candidates)
    print(f"Verifying {total} candidates against s3://{STORE_BUCKET}/{STORE_KEY_PREFIX} ...")

    progress = tqdm(total=total, unit="assembly", desc="Verifying checksums", leave=True)
    display(progress)

    def _update_progress(done: int, _total: int, acc: str) -> None:
        progress.n = done
        progress.set_postfix(acc=acc, refresh=False)
        progress.refresh()

    if total == 0:
        print("No candidates to verify; skipping checksum checks.")
        confirmed = set()
    else:
        confirmed = set(
            verify_transfer_candidates(
                candidates,
                filtered,
                STORE_BUCKET,
                STORE_KEY_PREFIX,
                ftp_host=FTP_HOST,
                progress_callback=_update_progress,
            )
        )

    progress.refresh()

    before = len(diff.new) + len(diff.updated)
    diff.new = [a for a in diff.new if a in confirmed]
    diff.updated = [a for a in diff.updated if a in confirmed]
    after = len(diff.new) + len(diff.updated)
    print(f"Verified: {after} need downloading, {before - after} pruned (already in store)")
else:
    print("Skipping S3 verification (STORE_BUCKET not set)")

# -- Apply LIMIT --
if LIMIT is not None:
    original_new = len(diff.new)
    original_updated = len(diff.updated)
    combined = diff.new + diff.updated
    limited = combined[:LIMIT]
    limited_set = set(limited)
    diff.new = [a for a in diff.new if a in limited_set]
    diff.updated = [a for a in diff.updated if a in limited_set]
    print(f"After limit ({LIMIT}): {len(diff.new)} new, {len(diff.updated)} updated")
    print(f"  (was {original_new} new, {original_updated} updated)")


In [ ]:
"""Write manifest files and upload snapshot to S3."""

# Write transfer manifest
transfer_path = OUTPUT_DIR / "transfer_manifest.txt"
paths = write_transfer_manifest(diff, filtered, transfer_path, ftp_host=FTP_HOST)
print(f"Transfer manifest: {len(paths)} entries -> {transfer_path}")

# Write removed manifest
removed_path = OUTPUT_DIR / "removed_manifest.txt"
removed = write_removed_manifest(diff, removed_path)
print(f"Removed manifest: {len(removed)} entries -> {removed_path}")

# Write updated manifest (for Phase 3 pre-overwrite archiving)
updated_path = OUTPUT_DIR / "updated_manifest.txt"
updated = write_updated_manifest(diff, updated_path)
print(f"Updated manifest: {len(updated)} entries -> {updated_path}")

# Write diff summary
summary_path = OUTPUT_DIR / "diff_summary.json"
summary = write_diff_summary(diff, summary_path, DATABASE, PREFIX_FROM, PREFIX_TO)
print(f"Diff summary -> {summary_path}")
print(json.dumps(summary["counts"], indent=2))

# Upload new snapshot to S3 for future diffing
if SNAPSHOT_UPLOAD_URI:
    s3 = get_s3_client()
    bucket, key = split_s3_path(SNAPSHOT_UPLOAD_URI)
    s3.put_object(Bucket=bucket, Key=key, Body=raw_summary.encode("utf-8"))
    print(f"Uploaded new snapshot to {SNAPSHOT_UPLOAD_URI}")
else:
    print("Skipping S3 snapshot upload (SNAPSHOT_UPLOAD_URI not set)")

In [ ]:
"""Upload manifests to S3 for CTS input staging (optional).

Note: STAGING_URI is a full s3:// URI. The promote notebook splits this into
STORE_BUCKET + STAGING_KEY_PREFIX (separate bucket and key prefix parameters).

This is for local testing. The CTS will stage the container's input folder in production.
"""

# S3 location where CTS will read input files from.
# Set to None to skip upload (local-only testing).
# format: s3:// URI  (e.g. "s3://cdm-lake/staging/run1/")
STAGING_URI: str | None = "s3://cdm-lake/staging/run1/input/"

if STAGING_URI:
    s3 = get_s3_client()
    bucket, prefix = split_s3_path(STAGING_URI)
    prefix = prefix.rstrip("/") + "/"

    for manifest in ["transfer_manifest.txt", "removed_manifest.txt", "updated_manifest.txt", "diff_summary.json"]:
        local_path = OUTPUT_DIR / manifest
        if local_path.exists():
            key = f"{prefix}{manifest}"
            s3.upload_file(Filename=str(local_path), Bucket=bucket, Key=key)
            print(f"Uploaded {manifest} -> s3://{bucket}/{key}")

    print(f"\nManifests staged for CTS at s3://{bucket}/{prefix}")
else:
    print("Skipping S3 manifest upload (STAGING_URI not set)")